# Clasificación usando un modelo preentrenado


In [ ]:
# La versión más reciente de la biblioteca transformers no incluye soporte para Keras
!pip install -q transformers==4.41.2 tensorflow==2.15.0

In [ ]:

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased"
max_len = 256
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

enc = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding="max_length",
    max_length=max_len,
    return_tensors="np"
)

X_ids  = enc["input_ids"]
X_mask = enc["attention_mask"]
model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
# ----------------------
# Phase 1: train only the classifier head
# ----------------------
model.distilbert.trainable = False  # freeze the encoder
model.pre_classifier.trainable = True
model.classifier.trainable = True

model.compile(optimizerzer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Phase 1: encoder frozen, training head")
history = model.fit(
    [X_ids, X_mask],
    y,
    epochs=1,
    validation_split=0.2)

plot_training_history(history)
# ----------------------
# Phase 2: unfreeze last N transformer blocks + head
# ----------------------
N = 2  # last 2 blocks out of 6 (nice illustrative choice)
model.distilbert.trainable = True

# Freeze early blocks, unfreeze last N
for block in model.distilbert.transformer.layer[:-N]:
    block.trainable = False
for block in model.distilbert.transformer.layer[-N:]:
    block.trainable = True

# Recompile after changing trainable flags
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),  # standard fine-tuning LR
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

print(f"Phase 2: fine-tuning last {N} blocks + head")
history = model.fit([X_ids, X_mask], y, batch_size=batch_size, epochs=3, shuffle=True)
plot_training_history(history)